# GeoFire-X : contrôle CPU du corpus train/validation
Notebook privé, accélérateur désactivé. Attacher le code et les huit lots après autorisation de publication. Aucun TIFF 2021. Aucun entraînement ou profilage GPU automatique.


In [ ]:
from pathlib import Path
import hashlib, json, os, subprocess, sys, zipfile
payload = Path('/kaggle/input/geofirex-expanded-code-recovery-v1/payload.zip')
with payload.open('rb') as stream:
    assert hashlib.file_digest(stream, 'sha256').hexdigest() == '7f942c6cccebe17096064f3a641f1b110a5048d48a73348bcc1cf6dc0aeeb9fe'
work = Path('/kaggle/working/geofirex-expanded-recovery')
work.mkdir(exist_ok=False)
with zipfile.ZipFile(payload) as archive:
    manifest = json.loads(archive.read('code-manifest.json'))
    assert len(archive.namelist()) == len(set(archive.namelist()))
    assert set(archive.namelist()) == set(manifest) | {'code-manifest.json'}
    for name in archive.namelist():
        target = (work/name).resolve()
        assert target.is_relative_to(work.resolve())
        data = archive.read(name)
        if name in manifest:
            assert hashlib.sha256(data).hexdigest() == manifest[name]
        target.parent.mkdir(parents=True, exist_ok=True)
        with target.open('xb') as stream:
            stream.write(data)
os.chdir(work)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-deps', '--no-build-isolation', '-e', '.'], check=True)
subprocess.run([sys.executable, '-c', 'import torch,pyarrow,psutil,yaml,mlflow'], check=True)


In [ ]:
# CPU : vérification des lots déjà attachés, aucune publication.
subprocess.run([sys.executable, 'scripts/mount_expanded_corpus.py', '--bundle', 'reports/recovery_20261107_v3/finalized/deployment/bundle',
                '--out', '/kaggle/working/expanded-mount-v1'], check=True)


In [ ]:
# Après autorisation GPU et activation manuelle de CUDA : B0/B1 d'abord.
# Ces commandes sont affichées, jamais exécutées par ce notebook CPU.
for model in ('B0', 'B1', 'B2', 'B3'):
    print(f"python scripts/profile_expanded_gpu.py --config /kaggle/working/expanded-mount-v1/{model}_T3_seed42.yaml --out /kaggle/working/profile-{model}-v1 --seconds 900 --steps 50 --warmup 5")
